# Value at Risk

Follow one risk number from market data to a plain-English answer. You read the TimesFM forecast, trace a Monte Carlo path, compare forward-looking and historical VaR, add a stress scenario, and ask the data agent about it.

This notebook is the same walkthrough as [`docs/01-value-at-risk.md`](https://github.com/gCloud-Tech-Showcase/datacloud-market-risk/blob/main/docs/01-value-at-risk.md), with charts. Run the cells in order. Your numbers differ slightly from the guide, because TimesFM does not always return identical intervals for identical input.

### Before you run it

| | |
| --- | --- |
| **Run time** | About 3 minutes top to bottom. Step 5 writes 3,000,000 rows in 10 to 20 seconds. Everything else reads tables the pipeline already built. |
| **Changes it makes** | Step 5 adds a `tech_down_10` scenario. **Reset**, at the end, removes it. Both are safe to run more than once. |
| **Charts** | Each step has one. The caption above it says what to look for. |

### Five terms you'll see

| Term | Plain meaning |
| --- | --- |
| **VaR (1-day, 99%)** | The loss the portfolio exceeds on only 1 day in 100 |
| **Expected Shortfall (ES)** | The average loss on those worst 1 in 100 days |
| **Volatility** | How much a price typically swings, here per year (annualized) |
| **Filtered Historical Simulation** | Replay real past days, rescaled to today's volatility |
| **Scenario** | A what-if shock applied on top of every simulated day, such as "Tech falls 10%" |

## Set up

In [ ]:
import google.auth
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import FuncFormatter

_, PROJECT_ID = google.auth.default()
print(f"Project: {PROJECT_ID}")

# Shared chart style: one colour per desk, dollar axes in millions.
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.titleweight": "bold", "axes.titlesize": 12, "axes.grid": True, "grid.alpha": 0.25,
})
DESK_COLORS = {"Growth Equities": "#1A73E8", "Value Equities": "#188038", "Defensive Equities": "#9AA0A6"}
usd_m = FuncFormatter(lambda v, _: f"{'−' if v < 0 else ''}${abs(v) / 1e6:,.1f}M")

## Before you start

Check the stress scenarios. A fresh deployment has two, `baseline` and `energy_down_15`. If `tech_down_10` is listed from an earlier run, run the **Reset** section at the end first.

In [ ]:
%%bigquery
SELECT scenario_id, description, sector, shock_pct
FROM `raw.stress_scenarios`
ORDER BY scenario_id

## Step 1: Look at the portfolio

Pulsar Capital, a fictional firm, holds 30 US large-cap stocks across three desks. The prices behind the positions are real.

In [ ]:
%%bigquery portfolio
SELECT desk, book, COUNT(*) AS positions, SUM(notional_usd) AS notional_usd
FROM `raw.portfolio_positions`
GROUP BY desk, book
ORDER BY desk, book

In [ ]:
p = portfolio.sort_values("notional_usd")
fig, ax = plt.subplots(figsize=(9, 3.2))
ax.barh(p["book"], p["notional_usd"], color=p["desk"].map(DESK_COLORS))
for y, (v, n) in enumerate(zip(p["notional_usd"], p["positions"])):
    ax.text(v, y, f"  ${v / 1e6:.0f}M · {n} stocks", va="center", fontsize=9)
ax.xaxis.set_major_formatter(usd_m)
ax.set_xlim(0, p["notional_usd"].max() * 1.35)
ax.set_title(f"${portfolio['notional_usd'].sum() / 1e6:.0f}M across 3 desks and 5 books")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in DESK_COLORS.values()],
          labels=list(DESK_COLORS), loc="lower right", frameon=False, fontsize=9)
plt.show()
portfolio

## Step 2: Read the TimesFM forecast

One `AI.FORECAST` query forecasts all 30 stocks. This is its raw output for JPMorgan. Step 1 is dated the day after the as-of date, even on a weekend: TimesFM infers a daily step, so read step 1 as the next trading day.

In [ ]:
%%bigquery jpm_forecast
SELECT DATE(forecast_timestamp) AS step_date,
       forecast_value,
       prediction_interval_lower_bound AS lower_99,
       prediction_interval_upper_bound AS upper_99,
       ai_forecast_status
FROM `serving.timesfm_forecast`
WHERE symbol = 'JPM'
ORDER BY forecast_timestamp

In [ ]:
jpm_forecast

In [ ]:
%%bigquery jpm_history
SELECT date, close
FROM `processed.daily_returns`
WHERE symbol = 'JPM'
ORDER BY date DESC
LIMIT 60

In [ ]:
history = jpm_history.sort_values("date")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history["date"], history["close"], label="Close (last 60 trading days)")
ax.plot(jpm_forecast["step_date"], jpm_forecast["forecast_value"], label="TimesFM median")
ax.fill_between(jpm_forecast["step_date"], jpm_forecast["lower_99"], jpm_forecast["upper_99"], alpha=0.3, label="99% interval")
ax.set_title("JPM: TimesFM forecast and its 99% interval")
ax.set_ylabel("Price (USD)")
ax.legend()
plt.show()

The simulation needs a volatility, not a price. `serving.timesfm_forward_params` turns the 99% interval for step 1 into a 1-day volatility: `ln(upper / lower) / (2 × 2.5758)`.

In [ ]:
%%bigquery vols
SELECT symbol, sector,
       ROUND(100 * timesfm_vol_annualized, 1) AS timesfm_vol_pct,
       ROUND(100 * historical_vol_annualized, 1) AS historical_vol_pct
FROM `serving.timesfm_forward_params`
ORDER BY sector, symbol

**Reading the chart:** each line runs from a stock's trailing one-year volatility (grey dot) to TimesFM's forward view (coloured dot). Blue means TimesFM expects a calmer day than last year, red a jumpier one. Stocks are sorted by the gap.

In [ ]:
v = vols.assign(diff=vols["timesfm_vol_pct"] - vols["historical_vol_pct"]).sort_values("diff")
calmer = int((v["diff"] < 0).sum())

fig, ax = plt.subplots(figsize=(9, 8))
for y, r in enumerate(v.itertuples()):
    color = "#1A73E8" if r.diff < 0 else "#D93025"
    ax.plot([r.historical_vol_pct, r.timesfm_vol_pct], [y, y], color=color, lw=2, alpha=0.6)
    ax.scatter(r.historical_vol_pct, y, color="#9AA0A6", s=30, zorder=3)
    ax.scatter(r.timesfm_vol_pct, y, color=color, s=40, zorder=3)
ax.set_yticks(range(len(v)), v["symbol"])
ax.set_xlabel("Annualized volatility (%)")
ax.set_title(f"TimesFM expects a calmer day than last year for {calmer} of {len(v)} stocks")
ax.legend(handles=[plt.Line2D([], [], marker="o", ls="", color="#9AA0A6", label="Trailing 1 year"),
                   plt.Line2D([], [], marker="o", ls="", color="#1A73E8", label="TimesFM, lower"),
                   plt.Line2D([], [], marker="o", ls="", color="#D93025", label="TimesFM, higher")],
          loc="lower right", frameon=False, fontsize=9)
plt.show()

For most stocks, TimesFM's band implies lower volatility than the trailing year. The demo uses the band as a volatility input. It does not claim the forward view is more accurate.

## Step 3: Trace one Monte Carlo path

Each of the 100,000 paths replays one of the last 1,000 trading days for all 30 stocks together, rescaled to today's TimesFM volatility. The paths are a smoothed bootstrap of those 1,000 days: the count makes the distribution smooth, not better informed. Path 42 replays one day. Its portfolio P&L is the sum of the 30 rows.

In [ ]:
%%bigquery path_42
SELECT historical_date, symbol, desk, notional_usd,
       ROUND(sim_log_return, 5) AS sim_log_return, ROUND(sim_pnl_usd, 2) AS sim_pnl_usd
FROM `serving.mc_simulated_pnl`
WHERE scenario_id = 'baseline' AND path_id = 42
ORDER BY symbol

In [ ]:
print(f"Path 42 replays {path_42['historical_date'].iloc[0]}. Portfolio P&L: ${path_42['sim_pnl_usd'].sum():,.0f}")
path_42.head(10)

In [ ]:
p42 = path_42.sort_values("sim_pnl_usd")
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(p42["symbol"], p42["sim_pnl_usd"], color=p42["desk"].map(DESK_COLORS))
ax.axvline(0, color="#5F6368", lw=0.8)
ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"${v / 1e3:,.0f}K"))
ax.set_title(f"Path 42: one replayed day, 30 stocks, portfolio P&L ${p42['sim_pnl_usd'].sum() / 1e6:+.2f}M")
ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=c) for c in DESK_COLORS.values()],
          labels=list(DESK_COLORS), loc="lower right", frameon=False, fontsize=9)
plt.show()

## Step 4: Compare forward-looking and historical VaR

VaR at 99% is the loss the portfolio exceeds on only 1 path in 100. Expected Shortfall (ES) is the average loss on those worst paths.

In [ ]:
%%bigquery
SELECT scenario_id, method,
       ROUND(var_95_usd) AS var_95_usd, ROUND(var_99_usd) AS var_99_usd,
       ROUND(es_99_usd) AS es_99_usd, ROUND(var_99_pct_notional, 2) AS var_99_pct
FROM `serving.portfolio_var_summary`
ORDER BY method, scenario_id

The distribution behind the baseline Monte Carlo numbers, with the 99% VaR and ES cut-offs marked:

In [ ]:
%%bigquery baseline_paths
SELECT path_id, SUM(sim_pnl_usd) AS pnl_usd
FROM `serving.mc_simulated_pnl`
WHERE scenario_id = 'baseline'
GROUP BY path_id

In [ ]:
pnl = baseline_paths["pnl_usd"]
var_99 = -pnl.quantile(0.01)
es_99 = -pnl[pnl <= -var_99].mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(pnl / 1e6, bins=200)
ax.axvline(-var_99 / 1e6, color="red", label=f"99% VaR: ${var_99 / 1e6:.2f}M")
ax.axvline(-es_99 / 1e6, color="black", linestyle="--", label=f"99% ES: ${es_99 / 1e6:.2f}M")
ax.set_title("Simulated 1-day portfolio P&L, 100,000 paths (baseline)")
ax.set_xlabel("P&L (USD millions)")
ax.set_ylabel("Paths")
ax.legend()
plt.show()

Historical VaR is higher. The days that drive it:

In [ ]:
%%bigquery
SELECT date, ROUND(pnl_usd) AS pnl_usd, worst_day_rank
FROM `serving.historical_pnl`
ORDER BY worst_day_rank
LIMIT 5

**Reading the chart:** every bar is one real day from the last 500, replayed on today's portfolio. The red line is the 99% historical VaR: only 5 of the 500 days lose more. The three biggest losses are labelled.

In [ ]:
%%bigquery hist
SELECT date, pnl_usd, worst_day_rank
FROM `serving.historical_pnl`
ORDER BY date

In [ ]:
hist["date"] = pd.to_datetime(hist["date"])
var_hist = -hist["pnl_usd"].quantile(0.01)
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(hist["date"], hist["pnl_usd"], width=1.0,
       color=["#D93025" if x < -var_hist else "#9AA0A6" for x in hist["pnl_usd"]])
ax.axhline(-var_hist, color="#D93025", ls="--", lw=1.2, label=f"99% historical VaR: ${var_hist / 1e6:.2f}M")
worst = hist.nsmallest(3, "pnl_usd").sort_values("date")
ax.annotate("3 worst days: " + ", ".join(f"{d:%b} {d.day}" for d in worst["date"]) + f" {worst['date'].iloc[0]:%Y}",
            (worst["date"].iloc[0], worst["pnl_usd"].min()), xytext=(40, -2), textcoords="offset points",
            fontsize=9, va="center", arrowprops={"arrowstyle": "->", "color": "#5F6368"})
ax.yaxis.set_major_formatter(usd_m)
ax.set_title("The last 500 trading days, replayed on today's portfolio")
ax.legend(loc="upper right", frameon=False, fontsize=9)
plt.show()

Two things separate the numbers. The historical view uses the last 500 trading days and the simulation draws from the last 1,000, so the April 2025 tariff selloff weighs more in the historical tail. That window difference is most of the gap. The simulation also rescales each day to TimesFM's volatility, which accounts for a smaller part.

## Step 5: Add a stress scenario and simulate it

Ask a what-if: Information Technology falls 10% tomorrow. The next cell adds the scenario and runs the simulation procedure for it, writing 3,000,000 rows. If the scenario already exists, it is replaced.

In [ ]:
%%bigquery
DELETE FROM `raw.stress_scenarios` WHERE scenario_id = 'tech_down_10';

INSERT INTO `raw.stress_scenarios` (scenario_id, description, sector, shock_pct)
VALUES ('tech_down_10', 'Information Technology falls 10%', 'Information Technology', -0.10);

CALL `serving.simulate_scenario`('tech_down_10');

In [ ]:
%%bigquery by_desk
SELECT scenario_id, desk, ROUND(var_99_usd) AS var_99_usd
FROM `serving.var_by_desk`
WHERE scenario_id IN ('baseline', 'tech_down_10')
ORDER BY desk, scenario_id

In [ ]:
d = by_desk.pivot(index="desk", columns="scenario_id", values="var_99_usd")
fig, ax = plt.subplots(figsize=(9, 4))
x = range(len(d))
ax.bar([j - 0.2 for j in x], d["baseline"], width=0.4, color="#9AA0A6", label="Baseline")
ax.bar([j + 0.2 for j in x], d["tech_down_10"], width=0.4, color="#D93025", label="Tech falls 10%")
for j, (b, s) in enumerate(zip(d["baseline"], d["tech_down_10"])):
    ax.text(j + 0.2, s, f"+${(s - b) / 1e6:.2f}M" if s - b > 1 else "no change",
            ha="center", va="bottom", fontsize=9, fontweight="bold" if s - b > 1 else "normal")
ax.set_xticks(list(x), d.index)
ax.yaxis.set_major_formatter(usd_m)
ax.set_title("99% VaR by desk: only the desk holding Tech moves")
ax.legend(frameon=False)
plt.show()
d

Only Growth Equities moves, because it holds all the Information Technology stocks. GOOGL is Communication Services under GICS, so this shock does not hit it. The other desks are unchanged to the dollar: every scenario replays the same draws on the same paths.

The shock is applied on top of every path, so this is VaR given the shock: baseline risk plus the shock's loss. It is not Basel stressed VaR, which recalibrates to a stressed period.

## Step 6: Ask the data agent

The data agent answers in plain English and shows the SQL it ran. You can also chat with it in the console under **BigQuery > Agents > Pulsar Capital market risk**. This cell asks it through the Conversational Analytics API.

In [ ]:
from google.auth.transport.requests import AuthorizedSession
from google.cloud import bigquery
from IPython.display import Markdown, display

QUESTION = "How does the tech_down_10 scenario change our 99% VaR by desk?"

credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
session = AuthorizedSession(credentials)
API = f"https://geminidataanalytics.googleapis.com/v1/projects/{PROJECT_ID}/locations/global"

# OpenTofu gives the agent a new ID whenever its definition changes, so find it
# by its display name.
agents = session.get(f"{API}/dataAgents").json().get("dataAgents", [])
agent = next(a["name"] for a in agents if a.get("displayName") == "Pulsar Capital market risk")

response = session.post(
    f"{API}:chat",
    json={
        "messages": [{"userMessage": {"text": QUESTION}}],
        "dataAgentContext": {"dataAgent": agent},
    },
    timeout=300,
)
response.raise_for_status()

# The agent returns its SQL as generatedSql when it writes a new query. When it
# starts from a verified query, it returns only the BigQuery job it ran, so the
# SQL is read from that job.
sql, answer = [], []
for message in response.json():
    system = message.get("systemMessage", {})
    data = system.get("data", {})
    if data.get("generatedSql"):
        sql.append(data["generatedSql"])
    if data.get("bigQueryJob"):
        job = data["bigQueryJob"]
        sql.append(bigquery.Client(project=PROJECT_ID).get_job(job["jobId"], location=job.get("location")).query)
    if system.get("text", {}).get("textType") == "FINAL_RESPONSE":
        answer.extend(system["text"].get("parts", []))

print("SQL the agent ran:")
print(sql[-1] if sql else "(none returned)")
display(Markdown("**Answer:** " + " ".join(answer)))

### Check the answer against the view

Change `QUESTION` above to *Which 5 stocks contribute the most tail risk?* and rerun it. The chart below reads the same view the agent uses, so you can check its answer. Each stock's share of the tail (99% ES) sits next to its share of the portfolio. A red bar much longer than its grey bar is concentration.

In [ ]:
%%bigquery contrib
SELECT symbol, desk, notional_usd, es_99_contribution_usd, share_of_es_99_pct
FROM `serving.var_contributions`
WHERE scenario_id = 'baseline'
ORDER BY es_99_contribution_usd DESC
LIMIT 10

In [ ]:
c = contrib.iloc[::-1]
notional_pct = 100 * c["notional_usd"] / 100_000_000
fig, ax = plt.subplots(figsize=(9, 4.5))
y = range(len(c))
ax.barh([j + 0.2 for j in y], c["share_of_es_99_pct"], height=0.4, color="#D93025", label="Share of 99% ES (tail loss)")
ax.barh([j - 0.2 for j in y], notional_pct, height=0.4, color="#9AA0A6", label="Share of notional")
ax.set_yticks(list(y), c["symbol"])
ax.set_xlabel("% of total")
top = contrib.iloc[0]
ax.set_title(f"{top['symbol']}: {100 * top['notional_usd'] / 1e8:.0f}% of the money, {top['share_of_es_99_pct']:.1f}% of the tail risk")
ax.legend(loc="lower right", frameon=False, fontsize=9)
plt.show()

## Reset

Remove the scenario you added, so the next run starts from the seeded `baseline` and `energy_down_15`.

In [ ]:
%%bigquery
DELETE FROM `serving.mc_simulated_pnl` WHERE scenario_id = 'tech_down_10';
DELETE FROM `raw.stress_scenarios` WHERE scenario_id = 'tech_down_10';